In [5]:
!pip install -qU gradio langchain langchain-community langchain-huggingface langchain-text-splitters pypdf faiss-cpu sentence-transformers bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 45.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 108.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 32.4 MB/s eta 0:00:0

In [ ]:
# 1. Clear out Kaggle's default conflicting packages
!pip uninstall -y transformers bitsandbytes accelerate sentence-transformers langchain langchain-community langchain-huggingface gradio numpy

# 2. Install the clean Digital Tribologist stack
!pip install -qU \
    "numpy>=2.0.0,<2.1.0" \
    "bitsandbytes>=0.46.1" \
    "transformers>=4.44.0,<5.0.0" \
    "accelerate>=0.33.0" \
    "sentence-transformers>=3.0.1" \
    "langchain==0.2.16" \
    "langchain-community==0.2.17" \
    "langchain-huggingface==0.0.3" \
    "langchain-text-splitters==0.2.4" \
    "pypdf>=4.3.1" \
    "faiss-cpu>=1.8.0.post1" \
    "gradio>=4.42.0"

In [ ]:
import sys

# --- 1. EMERGENCY PATCH: Fix for potential backend check errors ---
try:
    import transformers.integrations.bitsandbytes as bnb_integration
    if not hasattr(bnb_integration, "validate_bnb_backend_availability"):
        def dummy_validate(): return True
        bnb_integration.validate_bnb_backend_availability = dummy_validate
        print("✅ System Patch Applied: Backend validation hook secured.")
except Exception as e:
    print(f"⚠️ Patch status: {e}")

# --- 2. Standard Imports ---
import gradio as gr
import torch
import os
import traceback
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.vectorstores import FAISS

# --- 3. Initialize the RAG Model ---
print("🚀 Loading Models... Ensuring strict 4-bit quantization on T4 GPU.")

model_id = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    quantization_config=bnb_config,
)

text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True, 
    temperature=0.1, 
    repetition_penalty=1.1,
    return_full_text=False, 
    pad_token_id=tokenizer.eos_token_id 
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

vector_db = None

# --- 4. RAG Processing Functions ---
def process_document(pdf_file):
    global vector_db
    if pdf_file is None:
        return "Please upload a valid PDF document."
    
    try:
        loader = PyPDFLoader(pdf_file.name)
        documents = loader.load()
        
        # Chunking strategy optimized for research papers
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=800, 
            chunk_overlap=100
        )
        texts = text_splitter.split_documents(documents)
        
        vector_db = FAISS.from_documents(texts, embeddings)
        return f"Success! '{os.path.basename(pdf_file.name)}' ingested. System is ready."
    except Exception as e:
        return f"Error processing document: {str(e)}"

def expert_query(question):
    global vector_db
    if vector_db is None:
        return "Knowledge base is empty. Please upload a research paper first."
    
    try:
        # Retrieve top 3 context snippets
        docs = vector_db.similarity_search(question, k=3)
        context = "\n\n".join([f"Excerpt {i+1}:\n{doc.page_content}" for i, doc in enumerate(docs)])
        
        # Phi-3 Specific Prompt Template
        prompt = f"<|system|>\nYou are a senior materials and manufacturing expert. Answer based ONLY on the provided context. If the information is not present, state that you do not know.<|end|>\n<|user|>\nContext:\n{context}\n\nQuestion: {question}<|end|>\n<|assistant|>\n"
        
        outputs = text_generation_pipeline(prompt)
        answer = outputs[0]['generated_text'].strip()
        
        sources = set([str(doc.metadata.get('page', 'Unknown')) for doc in docs])
        return f"{answer}\n\n*(Sources: PDF Pages {', '.join(sources)})*"

    except Exception as e:
        return f"Inference Error: {str(e)}"

# --- 5. Gradio Dashboard UI ---
with gr.Blocks(theme=gr.themes.Soft()) as dashboard:
    gr.Markdown("# 🏭 Digital Tribologist & Manufacturing Expert")
    gr.Markdown("Consultant for Laser Processing, HEAs, and Surrogate Modeling.")
    
    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="Upload Research Paper (PDF)", file_types=[".pdf"])
            upload_btn = gr.Button("Ingest PDF")
            status = gr.Textbox(label="Status", interactive=False)
            
        with gr.Column(scale=2):
            query_input = gr.Textbox(label="Ask a Question", placeholder="e.g. Describe the Multi-Output Gaussian Process framework.")
            submit_btn = gr.Button("Analyze", variant="primary")
            output = gr.Textbox(label="Expert Recommendation", lines=10, interactive=False)

    upload_btn.click(fn=process_document, inputs=file_input, outputs=status)
    submit_btn.click(fn=expert_query, inputs=query_input, outputs=output)

# share=True is required in Kaggle to generate the public link you can click on
dashboard.launch(share=True, debug=True)

🚀 Loading Models... Ensuring strict 4-bit quantization on T4 GPU.


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_115/170217958.py:107: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as dashboard:


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://69434291bf77071f0f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
